# Session 3: SOLID Principles & AI Validation

> Apply the 5 core design principles to prevent technical debt and build maintainable systems.

## 1. Single Responsibility & Open/Closed

### Single Responsibility Principle (SRP)
**"A class should have only one reason to change."**

A *reason to change* = a stakeholder whose requirement could force a rewrite. If marketing changes the email template **and** the DB team changes the schema, a class that handles both will be broken twice. SRP says each class should be breakable by exactly one team/requirement.

**The class-explosion trade-off**
Applying SRP strictly does create more classes. This is intentional and manageable:
- More classes → each is shorter, simpler, and independently testable.
- Use an *orchestrator* (a thin coordinator class, or a function) to wire them together in one place. The orchestrator has no logic itself — it just calls the small pieces in order.
- Compare: one 200-line class that's hard to test vs. four 30-line classes that can each be unit-tested in isolation.

**Rule of thumb:** if you struggle to name a class with a single noun phrase (e.g., `UserEmailAndDatabaseManager`), it's doing too much.

### Open/Closed Principle (OCP)
**"Open for extension, closed for modification."**

Adding a new discount type should not require editing existing `if/elif` chains. Instead, create a new subclass of `PricingStrategy`. The calling code (`calculate(strategy, price)`) never changes. This is especially valuable in code that other teams depend on — they can extend without breaking your stable API.

In [ ]:
# ❌ SRP Violation — UserManager does too much
class UserManager:
    def create_user(self, data): pass
    def send_welcome_email(self, user): pass  # Not its job
    def save_to_database(self, user): pass    # Not its job

# ✅ SRP Compliant
class UserService:
    def create(self, data): return {'id': 1, **data}

class EmailService:
    def send_welcome(self, user): print(f'Email sent to {user["email"]}')

class UserRepository:
    def save(self, user): print(f'Saved user {user["id"]}')

# ✅ OCP — Open for extension, closed for modification
from abc import ABC, abstractmethod
class PricingStrategy(ABC):
    @abstractmethod
    def calculate(self, price: float) -> float: pass

class StandardPricing(PricingStrategy):
    def calculate(self, price): return price

class EnterpriseDiscount(PricingStrategy):
    def calculate(self, price): return price * 0.75

strategy = EnterpriseDiscount()
print(f'Enterprise price: ${strategy.calculate(1000)}')

## 💡 Additional Examples: SRP & OCP

The examples below show three common real-world scenarios where SOLID violations cause pain, and how to fix them.

**Example 1 — SRP: Report pipeline**
A single `ReportService` that generates, formats, and emails a report has three reasons to change: product changes the data model, design changes the PDF layout, ops changes the email provider. Splitting it into `ReportGenerator`, `PdfFormatter`, and `ReportMailer` means each change touches exactly one file. An orchestrator function sequences the three steps.

**Example 2 — OCP: Shape area calculator**
Adding a `Hexagon` shape should not require editing `calculate_total_area`. By making `Shape` an abstract base class, you can add unlimited shapes without touching existing code — the function loop `sum(s.area() for s in shapes)` works forever.

**Example 3 — LSP + DIP: Notification system**
`NotificationService` should work identically whether you inject `EmailSender`, `SmsSender`, or `SlackSender`. This is LSP: any subclass can substitute the base without surprising the caller. DIP is satisfied because `NotificationService.__init__` accepts the *abstraction* (`NotificationSender`), not a concrete type — so in tests you can pass a `FakeSender` that records calls instead of hitting real APIs.

In [ ]:
# ─── Example 1: SRP — Split Report processing into separate classes ───────────
# ❌ SRP Violation: ReportService handles generation, formatting, AND emailing
class ReportServiceBad:
    def generate(self, data): return {'rows': data, 'total': len(data)}
    def format_as_pdf(self, report): return f'[PDF bytes for {len(report["rows"])} rows]'
    def send_by_email(self, pdf, recipient): print(f'Emailing {len(pdf)} chars to {recipient}')

# ✅ SRP Compliant: each class has exactly one responsibility
class ReportGenerator:
    def generate(self, data: list) -> dict:
        return {'rows': data, 'total': len(data), 'avg': sum(data) / len(data) if data else 0}

class PdfFormatter:
    def format(self, report: dict) -> bytes:
        content = f"Total: {report['total']}, Avg: {report['avg']:.2f}"
        return content.encode()

class ReportMailer:
    def send(self, pdf_bytes: bytes, recipient: str):
        print(f'✉️  Sent {len(pdf_bytes)}-byte PDF to {recipient}')

# Orchestration layer wires the pieces together
data = [120, 340, 80, 220]
report = ReportGenerator().generate(data)
pdf = PdfFormatter().format(report)
ReportMailer().send(pdf, 'manager@company.com')

# ─── Example 2: OCP — Shape Area Calculator ──────────────────────────────────
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, radius: float): self.radius = radius
    def area(self) -> float: return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, w: float, h: float): self.w, self.h = w, h
    def area(self) -> float: return self.w * self.h

class Triangle(Shape):
    def __init__(self, base: float, height: float): self.base, self.height = base, height
    def area(self) -> float: return 0.5 * self.base * self.height

# Adding a new shape never requires modifying calculate_total_area
def calculate_total_area(shapes: list[Shape]) -> float:
    return sum(s.area() for s in shapes)

shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]
for s in shapes:
    print(f'{s.__class__.__name__}: area = {s.area():.2f}')
print(f'Total area: {calculate_total_area(shapes):.2f}')

# ─── Example 3: LSP + DIP — Notification System ──────────────────────────────
class NotificationSender(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> bool: ...

class EmailSender(NotificationSender):
    def send(self, recipient, message):
        print(f'📧 Email → {recipient}: {message}')
        return True

class SmsSender(NotificationSender):
    def send(self, recipient, message):
        print(f'📱 SMS → {recipient}: {message[:160]}')  # SMS character limit
        return True

class SlackSender(NotificationSender):
    def send(self, recipient, message):
        print(f'💬 Slack → #{recipient}: {message}')
        return True

# DIP: NotificationService depends on abstraction, not on concrete implementations
class NotificationService:
    def __init__(self, sender: NotificationSender):
        self._sender = sender

    def notify(self, recipient: str, message: str) -> bool:
        return self._sender.send(recipient, message)

# LSP: any subclass can substitute NotificationSender without breaking behavior
for sender in [EmailSender(), SmsSender(), SlackSender()]:
    svc = NotificationService(sender)
    svc.notify('devteam', 'Deployment completed successfully 🚀')


## 2. LSP, ISP, DIP

### Liskov Substitution Principle (LSP)
**"Subclasses must be substitutable for their base class without changing the correctness of the program."**

The classic failure: `Square extends Rectangle`. Callers that do `rect.set_width(5); rect.set_height(10); assert rect.area() == 50` will break when a `Square` is passed, because a square must keep width == height. The fix: model them separately, or use a shared `Shape` base that has only what both truly share.

The practical test: if you need an `if isinstance(x, SubClass)` check inside code that is supposed to work on the base class, LSP is violated.

### Interface Segregation Principle (ISP)
**"No class should be forced to implement methods it does not use."**

In Python this surfaces when a base class defines 6 abstract methods but a concrete class only needs 2 — it has to implement the other 4 as `raise NotImplementedError`. The fix: split the interface into focused, small protocols. IoT devices are a classic example: a thermostat should never have to stub out a `take_photo` method.

### Dependency Inversion Principle (DIP)
**"High-level modules must not depend on low-level modules. Both should depend on abstractions."**

Practical impact: if `UserService` imports `PostgresUserRepository` directly, you cannot unit-test `UserService` without a running database. By depending on `UserRepositoryPort` (an ABC), you can inject `InMemoryUserRepository` in tests and `PostgresUserRepository` in production — the service code is identical in both cases. This is the foundation of testable architecture.

**Liskov Substitution (LSP):** Subclasses must be substitutable for their parent without breaking behavior.

**Interface Segregation (ISP):** No class should be forced to implement interfaces it doesn't use. Split large interfaces.

**Dependency Inversion (DIP):** Depend on abstractions, not concretions. High-level modules must not depend on low-level modules.

In [ ]:
# ─── Example: LSP — Bird Hierarchy ───────────────────────────────────────────
from abc import ABC, abstractmethod

class Bird(ABC):
    @abstractmethod
    def move(self) -> str: ...

class FlyingBird(Bird):
    def move(self) -> str: return 'flying'

class SwimmingBird(Bird):
    def move(self) -> str: return 'swimming'

class Eagle(FlyingBird):
    def move(self): return f'Eagle is {super().move()} at high altitude'

class Penguin(SwimmingBird):
    def move(self): return f'Penguin is {super().move()} in cold water'

# LSP: every subclass substitutes Bird without breaking caller behavior
for bird in [Eagle(), Penguin()]:
    print(bird.move())  # ✅ No NotImplementedError like the classic Square(Rectangle) trap

# ─── Example: ISP — IoT Device Interfaces ────────────────────────────────────
# ❌ ISP Violation: one bloated interface forces all devices to implement everything
class SmartDeviceBloated(ABC):
    @abstractmethod
    def get_temperature(self): ...
    @abstractmethod
    def take_photo(self): ...   # A thermostat has no camera!
    @abstractmethod
    def unlock_door(self): ...  # A camera does not unlock doors!

# ✅ ISP Compliant: split into focused capability interfaces
class TemperatureSensor(ABC):
    @abstractmethod
    def get_temperature(self) -> float: ...

class Camera(ABC):
    @abstractmethod
    def take_photo(self) -> bytes: ...

class SmartLock(ABC):
    @abstractmethod
    def unlock(self, pin: str) -> bool: ...

class ThermostatDevice(TemperatureSensor):
    def get_temperature(self) -> float: return 22.5

class SecurityCamera(Camera, TemperatureSensor):
    def take_photo(self) -> bytes: return b'<jpeg data>'
    def get_temperature(self) -> float: return 28.0

thermostat = ThermostatDevice()
camera = SecurityCamera()
print(f'Thermostat temp: {thermostat.get_temperature()}°C')
print(f'Camera temp: {camera.get_temperature()}°C, photo: {len(camera.take_photo())} bytes')

# ─── Example: DIP — Repository Pattern ───────────────────────────────────────
class UserRepositoryPort(ABC):
    @abstractmethod
    def find_by_id(self, user_id: int) -> dict | None: ...
    @abstractmethod
    def save(self, user: dict) -> dict: ...

class InMemoryUserRepository(UserRepositoryPort):
    def __init__(self): self._store = {}
    def find_by_id(self, user_id): return self._store.get(user_id)
    def save(self, user):
        self._store[user['id']] = user
        return user

# Simulated production repository (PostgreSQL)
class PostgresUserRepository(UserRepositoryPort):
    def find_by_id(self, user_id): return {'id': user_id, 'name': 'DB User', 'source': 'postgres'}
    def save(self, user): print(f'INSERT INTO users ... {user["name"]}'); return user

# UserService does not know (or care) which backend is used
class UserService:
    def __init__(self, repo: UserRepositoryPort):  # ← Depend on abstraction
        self._repo = repo

    def get_or_create(self, user_id: int, name: str) -> dict:
        user = self._repo.find_by_id(user_id)
        if not user:
            user = self._repo.save({'id': user_id, 'name': name})
        return user

# Use InMemory for tests, Postgres in production — service code stays identical
svc = UserService(InMemoryUserRepository())
u = svc.get_or_create(1, 'Alice')
print(f'Created: {u}')
u_again = svc.get_or_create(1, 'Alice')
print(f'Fetched again: {u_again}')


## 3. AI Lab: Codebase SOLID Audit

### 🧪 AI Lab / Practice

> **TODO:** Pick a class from your current project (50+ lines) → Paste into Claude with: 'Analyze for SOLID violations. List each violation with: principle violated, line number, why it violates, and a refactored version.' → Fix the top 2 violations and submit the diff.

In [ ]:
# ❌ E-commerce Order Processor
'''AI phân tích vi phạm SOLID trong code xử lý đơn hàng:
1. Vi phạm SRP (Đơn nhiệm): Class OrderProcessor đang làm quá nhiều việc:
vừa tính giá đơn hàng, vừa lo kết nối DB để lưu trữ, lại vừa lo cấu hình gửi email.
Nếu ngày mai đội vận hành muốn đổi từ gửi Email sang gửi tin nhắn SMS, bạn sẽ phải sửa class này.
2. Vi phạm OCP (Đóng/Mở): Nếu tháng sau sếp muốn thêm loại đơn hàng "BLACK_FRIDAY" để giảm 50%, bạn buộc phải nhảy vào sửa mạch if/elif.
Việc này rất dễ làm gõ nhầm hoặc ảnh hưởng đến công thức tính giá "VIP" hay "STUDENT" đang chạy ổn định.
3. Vi phạm DIP (Đảo ngược phụ thuộc): Code đang gắn chặt (hardcode) việc lưu vào MySQL.
Nếu dự án muốn chuyển sang lưu vào MongoDB hoặc khi bạn muốn viết Unit Test (chạy thử code mà không muốn lưu thật vào database),
bạn sẽ không thể làm được vì dòng chữ Kết nối tới MySQL Database... đã bị dính chặt vào logic xử lý đơn hàng.'''
class OrderProcessor:
    def process_orders(self, order_type: str, amount: float, customer_email: str):
        # 1. Xử lý tính toán dựa trên loại đơn hàng (Vi phạm OCP)
        if order_type == "normal":
            final_amount = amount
        elif order_type == "vip":
            final_amount = amount * 0.9  # Giảm 10%
        elif order_type == "student":
            final_amount = amount * 0.8  # Giảm 20%
        else:
            final_amount = amount

        print(f"Tính toán xong tiền cho đơn hàng: {final_amount}")

        # 2. Kết nối database và lưu đơn hàng (Vi phạm SRP & DIP)
        # Kết nối trực tiếp tới MySQL database cấp thấp
        print(f"Kết nối tới MySQL Database...")
        print(f"Lưu đơn hàng trị giá {final_amount} vào bảng `orders` thành công.")

        # 3. Gửi email xác nhận cho khách hàng (Vi phạm SRP)
        print(f"Kết nối tới SMTP Server...")
        print(f"Gửi mail tới {customer_email}: Đơn hàng của bạn đã được xử lý!")
#✅ Refactor E-commerce Order Processor
#SRP
class NotificationService:
    def send_confirmation(self, email: str, msg: str):
        print(f"Gửi mail tới {email}: {msg}")

# OCP: 
from abc import ABC, abstractmethod
class PricingStrategy(ABC):
    @abstractmethod
    def calculate(self, amount: float) -> float: ...
    pass
class NormalPricing(PricingStrategy):
    def calculate(self, amount: float) -> float:
        return amount
class VipPricing(PricingStrategy):
    def calculate(self, amount: float) -> float:
        return amount * 0.9
class StudentPricing(PricingStrategy):
    def calculate(self, amount: float) -> float:
        return amount * 0.8

#DIP
class OrderRepositoryPort(ABC):
    @abstractmethod
    def save_order(self, amount: float) -> float:
        pass

class MySqlOrderRepository(OrderRepositoryPort):
    def save_order(self, amount: float) -> float:
        print(f"Kết nối tới MySQL Database...")
        print(f"Lưu đơn hàng trị giá {amount} vào bảng `orders` thành công.")
        return amount
class InMemoryOrderRepository(OrderRepositoryPort):
    def __init__(self):
        self.orders = []
    def save_order(self, amount: float) -> float:
        self.orders.append(amount)
        print(f"Lưu đơn hàng trị giá {amount} vào bộ nhớ thành công.")
        return amount

class OrderProcessor:
    def __init__(self, repo: OrderRepositoryPort, notifier: NotificationService):
        self._repo = repo          # Phụ thuộc vào abstraction(DIP)
        self._notifier = notifier  # SRP

    def process(self, strategy: PricingStrategy, amount: float, email: str):
        # 1. Tính giá dựa vào chiến lược được truyền vào (Thỏa mãn OCP)
        final_amount = strategy.calculate(amount)
        
        # 2. Lưu trữ đơn hàng qua cổng trừu tượng (Thỏa mãn SRP & DIP)
        self._repo.save_order(final_amount)
        
        # 3. Gửi thông báo (Thỏa mãn SRP)
        self._notifier.send_confirmation(email, "Đơn hàng của bạn đã được xử lý!")

# Khởi tạo các công cụ dùng chung
notifier = NotificationService()

# TÌNH HUỐNG 1: Chạy thật ở Production (Dùng MySQL + Giá VIP)
print("--- CHẠY THẬT TẠI PRODUCTION ---")
prod_repo = MySqlOrderRepository()
processor_prod = OrderProcessor(prod_repo, notifier)

vip_strategy = VipPricing()
processor_prod.process(vip_strategy, 1000.0, "khach_vip@gmail.com")


# TÌNH HUỐNG 2: Chạy thử nghiệm Unit Test (Không bật DB thật, dùng giá Student)
print("\n--- CHẠY TRONG MÔI TRƯỜNG UNIT TEST ---")
test_repo = InMemoryOrderRepository()
processor_test = OrderProcessor(test_repo, notifier)

student_strategy = StudentPricing()
processor_test.process(student_strategy, 1000.0, "sinhvien@university.edu")

In [ ]:

# ══════════════════════════════════════════════════════════════════
# BÀI 1 — Single Responsibility Principle (SRP)
# ══════════════════════════════════════════════════════════════════

# ── CODE GỐC ──────────────────────────────────────────────────────
class UserManager:
    def __init__(self, db_url: str):
        self.db_url = db_url

    def register(self, data: dict) -> dict:
        # Validate
        if not data.get('email') or '@' not in data['email']:
            raise ValueError("Invalid email")
        if not data.get('password') or len(data['password']) < 8:
            raise ValueError("Password too short")

        # Create user
        user = {'id': 1, 'email': data['email'], 'role': 'member'}

        # Save to DB
        print(f"[DB] INSERT INTO users (email) VALUES ('{user['email']}')")

        # Send welcome email
        print(f"[SMTP] Sending welcome email to {user['email']}")

        # Audit log
        print(f"[AUDIT] New user registered: id={user['id']}")

        return user

# ── CÂU HỎI ───────────────────────────────────────────────────────
# 1. Class này vi phạm nguyên tắc nào? Có bao nhiêu "lý do để thay đổi"?
#    Liệt kê từng lý do và team nào sẽ yêu cầu thay đổi.
#
# 2. Nếu team Ops đổi SMTP → SendGrid, team DB đổi schema bảng users,
#    team Security thêm trường vào audit log — bao nhiêu chỗ trong file
#    này bị ảnh hưởng?
#
# 3. Refactor lại: tách thành các class riêng biệt, mỗi class
#    một trách nhiệm. Tên class nên phản ánh đúng trách nhiệm.

# ── TODO ──────────────────────────────────────────────────────────
'''
1. class vi phạm SRP vì có 4 lý do để thay đổi:
- Team Validation: có thể yêu cầu thay đổi logic validate email hoặc password.
- Team DB: có thể yêu cầu thay đổi cách lưu user vào database, hoặc đổi database từ MySQL sang MongoDB.
- Team Ops: có thể yêu cầu thay đổi cách gửi email, ví dụ chuyển từ SMTP sang SendGrid.
- Team Security: có thể yêu cầu thay đổi format hoặc nội dung của audit log.
2. toàn bộ class bị xung đột code
'''
# 3. Refactor
class UserRegister:
    def register(self, data: dict) -> dict:
        # Validate
        if not data.get('email') or '@' not in data['email']:
            raise ValueError("Invalid email")
        if not data.get('password') or len(data['password']) < 8:
            raise ValueError("Password too short")

        # Create user
        user = {'id': 1, 'email': data['email'], 'role': 'member'}
        return user

class UserRepository:
    def __init__(self, db_url: str):
        self.db_url = db_url

    def save(self, user: dict) -> None:
        print(f"[DB] INSERT INTO users (email) VALUES ('{user['email']}')")

class EmailService:
    def send_welcome_email(self, email: str) -> None:
        print(f"[SMTP] Sending welcome email to {email}")

class AuditLogger:
    def log_user_registration(self, user_id: int) -> None:
        print(f"[AUDIT] New user registered: id={user_id}")

class UserManager:
    def __init__(self, db_url: str):
        self.user_register = UserRegister()
        self.user_repo = UserRepository(db_url)
        self.email_service = EmailService()
        self.audit_logger = AuditLogger()

    def register(self, data: dict) -> dict:
        user = self.user_register.register(data)
        self.user_repo.save(user)
        self.email_service.send_welcome_email(user['email'])
        self.audit_logger.log_user_registration(user['id'])
        return user

[DB] INSERT INTO users (email) VALUES ('intern@company.com')
[SMTP] Sending welcome email to intern@company.com
[AUDIT] New user registered: id=1

🎉 Đăng ký thành công: {'id': 1, 'email': 'intern@company.com', 'role': 'member'}

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BÀI 2 — Open/Closed Principle (OCP)
# ══════════════════════════════════════════════════════════════════

import math
from abc import ABC, abstractmethod

# ── CODE GỐC ──────────────────────────────────────────────────────
class Circle_Bad:
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Rectangle_Bad:
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

def get_area_description_bad(shape) -> str:
    if isinstance(shape, Circle_Bad):
        return f"Circle area: {shape.area():.2f}"
    elif isinstance(shape, Rectangle_Bad):
        return f"Rectangle area: {shape.area():.2f}"
    else:
        return "Unknown shape"

# ── CÂU HỎI ───────────────────────────────────────────────────────
# 1. Hàm get_area_description_bad vi phạm nguyên tắc nào?
#    Dấu hiệu cụ thể nào trong code cho thấy vi phạm đó?
#
# 2. Nếu thêm class Triangle, Hexagon, Ellipse —
#    hàm này phải thay đổi bao nhiêu lần? Đây có phải vấn đề không?
#
# 3. Viết lại để khi thêm Triangle(Shape), hàm get_area_description
#    KHÔNG CẦN SỬA. Shape phải có gì để đảm bảo điều này?

# ── TODO ───────────────────────────────────────────────────────
'''
1. hàm get_area_description_bad vi phạm nguyễn tắc OCP
Dấu hiệu: hàm dùng nhiều câu lệnh if/elif isinstance
2. nếu thêm class Triangle, Hexagon, Ellipse — hàm này phải thay đổi 3 lần.
Đây là vấn đề vì dễ làm gõ nhầm hoặc ảnh hưởng đến code đang chạy ổn định.
'''
# 3. Refactor
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):
        return math.pi * self.r ** 2
    
class Rectangle(Shape):
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h
    
class Triangle(Shape):
    def __init__(self, base, height):
        self.base, self.height = base, height
    def area(self):
        return 0.5 * self.base * self.height
    
def get_area_description(shape: Shape) -> str:
    return f"{shape.__class__.__name__} area: {shape.area():.2f}"

In [ ]:
# ══════════════════════════════════════════════════════════════════
# BÀI 3 — LSP + ISP (Liskov Substitution + Interface Segregation)
# ══════════════════════════════════════════════════════════════════

# ── CODE GỐC ──────────────────────────────────────────────────────
class Animal_Bad(ABC):
    @abstractmethod
    def speak(self) -> str: ...

    @abstractmethod
    def fly(self) -> str: ...


class Dog_Bad(Animal_Bad):
    def speak(self): return "Woof!"
    def fly(self): raise NotImplementedError("Dogs cannot fly")


class Bird_Bad(Animal_Bad):
    def speak(self): return "Tweet!"
    def fly(self): return "Flapping wings..."


def make_it_fly(animal: Animal_Bad) -> str:
    return animal.fly()

# ── CÂU HỎI ───────────────────────────────────────────────────────
# 1. Code trên vi phạm đồng thời 2 nguyên tắc SOLID. Đó là những
#    nguyên tắc nào? Giải thích từng nguyên tắc bị vi phạm như thế nào.
#
# 2. make_it_fly(Dog_Bad()) gây ra lỗi gì? Tại sao đây là vi phạm LSP?
#    (Gợi ý: "subclass must be substitutable for their base class")
#
# 3. Refactor lại toàn bộ hierarchy:
#    - Không class nào phải implement method nó không có khả năng thực hiện.
#    - Dog và Bird vẫn dùng chung được trong code xử lý speak().
#    - Chỉ Bird (và các loài biết bay) mới có method fly().

# ── TODO ──────────────────────────────────────────────────────────
'''
1. Code trên vi phạm đồng thời 2 nguyên tắc SOLID:
- Vi phạm LSP (Liskov Substitution Principle):
  Dog_Bad không thể thay thế Animal_Bad trong mọi ngữ cảnh, vì nó không thể thực hiện phương thức fly().
- Vi phạm ISP (Interface Segregation Principle):
  Animal_Bad định nghĩa cả hai phương thức speak() và fly(), nhưng không phải tất cả các loài động vật đều có khả năng bay.
  Điều này buộc Dog_Bad phải implement fly() dù nó không có khả năng bay, dẫn đến việc phải raise NotImplementedError.
2. make_it_fly(Dog_Bad()) gây ra lỗi NotImplementedError vì Dog_Bad không thể thực hiện phương thức fly().
Đây là vi phạm LSP vì Dog_Bad không thể thay thế Animal_Bad trong mọi ngữ cảnh mà Animal_Bad được sử dụng,
do nó không hỗ trợ tất cả các phương thức được định nghĩa trong Animal_Bad.
'''
# 3. Refactor
from abc import ABC, abstractmethod

class Speaker(ABC):
    @abstractmethod
    def speak(self) -> str: ...

class Flyer(ABC):
    @abstractmethod
    def fly(self) -> str: ...

class Dog(Speaker):
    def speak(self):
        return "Woof!"

class Bird(Speaker, Flyer):
    def speak(self): 
        return "Tweet!"
    
    def fly(self): 
        return "Flapping wings..."

def make_it_fly(flyer: Flyer) -> str:
    return flyer.fly()

def make_it_speak(speaker: Speaker) -> str:
    return speaker.speak()
